In [ ]:
# CÉLULA 1: CONECTAR DRIVE E PREPARAR COMFYUI

from google.colab import drive
import os
import shutil
import subprocess

print("🔄 Conectando ao Google Drive...")
drive.mount("/content/drive")

DRIVE_PATH = "/content/drive/MyDrive/ComfyUI"
TEMP_PATH = "/content/ComfyUI_install"

# ---------------------------------------------------------
# 1. INSTALAR / ATUALIZAR COMFYUI
# ---------------------------------------------------------

if not os.path.exists(os.path.join(DRIVE_PATH, "main.py")):
    print("📥 ComfyUI não encontrado. Instalando...")

    # Como a pasta ComfyUI pode já existir com models/custom_nodes,
    # fazemos o clone temporariamente fora do Drive.
    if os.path.exists(TEMP_PATH):
        shutil.rmtree(TEMP_PATH)

    !git clone https://github.com/Comfy-Org/ComfyUI.git {TEMP_PATH}

    # Cria a pasta no Drive se necessário
    os.makedirs(DRIVE_PATH, exist_ok=True)

    # Mescla o ComfyUI com o conteúdo que já existe no Drive.
    # Isso preserva models/custom_nodes existentes.
    shutil.copytree(
        TEMP_PATH,
        DRIVE_PATH,
        dirs_exist_ok=True
    )

    shutil.rmtree(TEMP_PATH)

    print("✅ ComfyUI instalado.")

else:
    print("✅ ComfyUI detectado. Atualizando...")

    %cd {DRIVE_PATH}

    # Verifica se é um repositório git válido
    if os.path.exists(os.path.join(DRIVE_PATH, ".git")):
        !git pull --ff-only
    else:
        print("⚠️ A instalação existe, mas não possui .git.")
        print("   Pulando atualização automática.")

# ---------------------------------------------------------
# 2. CRIAR PASTAS DE MODELOS
# ---------------------------------------------------------

os.makedirs(
    os.path.join(DRIVE_PATH, "models", "checkpoints"),
    exist_ok=True
)

os.makedirs(
    os.path.join(DRIVE_PATH, "models", "loras"),
    exist_ok=True
)

os.makedirs(
    os.path.join(DRIVE_PATH, "custom_nodes"),
    exist_ok=True
)

# ---------------------------------------------------------
# 3. INSTALAR DEPENDÊNCIAS
# ---------------------------------------------------------

%cd {DRIVE_PATH}

print("📦 Instalando dependências do ComfyUI...")

!pip install -q -r requirements.txt

# Manager oficial do ComfyUI atual
if os.path.exists("manager_requirements.txt"):
    print("📦 Instalando dependências do ComfyUI Manager...")
    !pip install -q -r manager_requirements.txt

print("")
print("✅ Célula 1 concluída!")
print(f"📁 ComfyUI: {DRIVE_PATH}")

In [ ]:
# CÉLULA 2: AUTENTICAR NGROK E LIGAR COMFYUI
!pip install pyngrok
from pyngrok import ngrok
import getpass

%cd /content/drive/MyDrive/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!pip install -r ComfyUI-Manager/requirements.txt

print("🔑 Cole seu Authtoken do Ngrok:")
NGROK_TOKEN = getpass.getpass("Token: ")

# Remove o arquivo de configuração antigo, se existir
config_path = "/root/.config/ngrok/ngrok.yml"
if os.path.exists(config_path):
    os.remove(config_path)

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8188)

print(f"\n🚀 LINK DE ACESSO: {public_url}")

%cd /content/drive/MyDrive/ComfyUI
!python main.py --listen 127.0.0.1 --port 8188

In [ ]:
!find /content/drive/MyDrive/ComfyUI/models -maxdepth 2 -type f
!ls -ld /content/ComfyUI/models